# PPE Detection Model Training
Train YOLOv8m on your PPE dataset using free GPU

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics
!pip install ultralytics -q

In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 4: Unzip Dataset
!cp /content/drive/MyDrive/ppe_dataset.zip /content/
!unzip -q /content/ppe_dataset.zip -d /content/ppe_dataset
!ls /content/ppe_dataset

In [ ]:
# Step 5: Create data.yaml with correct paths
data_yaml = '''path: /content/ppe_dataset
train: images/train
val: images/val
test: images/test

nc: 5
names:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
'''

with open('/content/ppe_dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print('data.yaml created!')

In [ ]:
# Step 6: Train the Model (YOLOv8m)
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data='/content/ppe_dataset/data.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    patience=20,
    device=0,
    project='/content/runs',
    name='ppe_training',
    exist_ok=True,
    pretrained=True
)

In [ ]:
# Step 7: Check Results
from IPython.display import Image, display
import os

results_dir = '/content/runs/ppe_training'
for img_name in ['results.png', 'confusion_matrix.png']:
    img_path = os.path.join(results_dir, img_name)
    if os.path.exists(img_path):
        display(Image(filename=img_path, width=800))

In [ ]:
# Step 8: Test on Sample Image
import glob

best_model = YOLO('/content/runs/ppe_training/weights/best.pt')
test_images = glob.glob('/content/ppe_dataset/images/test/*')[:3]
for img_path in test_images:
    results = best_model(img_path, conf=0.25, verbose=False)
    print(f'
{os.path.basename(img_path)}:')
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        name = best_model.names[cls_id]
        print(f'  {name}: {conf:.2%}')

In [ ]:
# Step 9: Save best.pt to Google Drive
!cp /content/runs/ppe_training/weights/best.pt /content/drive/MyDrive/best.pt
print('Model saved to Google Drive as best.pt')
print('Download it and place in: D:/workers_Safety/models/best.pt')